# 13 — Áudio de fundo para a demonstração (eventos 3 e 4 da Tabela B2)

Gera um WAV contínuo com gravações **normais** de outros ventiladores do MIMII (id_02, id_04, id_06), os mesmos usados como fundo nas sessões (notebook 07b), para tocar no alto-falante durante os eventos com ruído de fundo.

* Usa só clipes da partição **teste** do MIMII, de modo que o fundo da demonstração não repete gravações usadas no treino.
* Transição suave de 50 ms entre clipes, sem cortes audíveis.
* Nível normalizado com folga de pico (sem saturação); o volume final é ajustado no alto-falante até **70 dB(A) no ponto do sensor** (Apêndice B, item e).

**Saídas** em `dissertacao_tinyml/demonstracao/`: `fundo_demonstracao.wav` e `fundo_demonstracao.json` (lista dos clipes, para o guia de replicação).

In [1]:
# 1. Ambiente e caminhos
import os, io, json, zipfile
import numpy as np, pandas as pd, soundfile as sf
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/dissertacao_tinyml"
except ImportError:
    BASE = os.environ.get("BASE_TINYML", "dissertacao_tinyml")
ZIP = f"{BASE}/mimii_fan_6db/6_dB_fan.zip"
PARTICAO = f"{BASE}/mimii_fan_6db/particao_por_gravacao.csv"
SAIDA = f"{BASE}/demonstracao"; os.makedirs(SAIDA, exist_ok=True)

Mounted at /content/drive


In [2]:
# 2. Parâmetros
DURACAO_MIN = float(os.environ.get("DURACAO_FUNDO_MIN", 20))   # duração do arquivo
MAQUINAS = ["id_02", "id_04", "id_06"]   # mesmas do fundo das sessões (parametros_geracao.json)
PARTICAO_USADA = "teste"
NIVEL_RMS_DBFS = -20.0                   # nível médio; o volume final é ajustado no alto-falante
TRANSICAO_MS = 50
SEMENTE = 2026
rng = np.random.default_rng(SEMENTE)

In [3]:
# 3. Clipes normais dos ventiladores de fundo, só da partição de teste
inv = pd.read_csv(PARTICAO)
sel = inv[inv.maquina.isin(MAQUINAS) & (inv.rotulo == 0) & (inv.particao == PARTICAO_USADA)]
print("Clipes disponíveis por máquina:", sel.maquina.value_counts().to_dict())
assert len(sel), "nenhum clipe encontrado: confira o particao_por_gravacao.csv"
z = zipfile.ZipFile(ZIP)
FS = 16000
n_alvo = int(DURACAO_MIN * 60 * FS); tr = int(TRANSICAO_MS / 1000 * FS); r = np.linspace(0, 1, tr, dtype=np.float32)
fila, usados, out = list(rng.permutation(sel.membro.values)), [], np.zeros(0, np.float32)
while len(out) < n_alvo + tr:
    if not fila: fila = list(rng.permutation(sel.membro.values))   # recomeça só se o conjunto acabar
    m = fila.pop(); usados.append(m)
    x, fs = sf.read(io.BytesIO(z.read(m)), dtype="float32", always_2d=True); assert fs == FS
    x = x[:, 0]
    x = x / (np.sqrt(np.mean(x.astype(np.float64) ** 2)) + 1e-12)   # cada clipe com o mesmo RMS: sem saltos de volume
    out = x if not len(out) else np.concatenate([out[:-tr], out[-tr:] * (1 - r) + x[:tr] * r, x[tr:]])
out = out[:n_alvo]
print(f"{len(usados)} clipes, {len(set(usados))} distintos, {len(out) / FS / 60:.1f} min")

Clipes disponíveis por máquina: {'id_04': 155, 'id_02': 153, 'id_06': 153}
121 clipes, 121 distintos, 20.0 min


In [4]:
# 4. Nível, fade de entrada e saída, gravação
out = out * 10 ** (NIVEL_RMS_DBFS / 20)
pico = np.abs(out).max()
if pico > 0.89: out *= 0.89 / pico                  # folga de 1 dB abaixo do máximo
f = np.linspace(0, 1, 2 * FS, dtype=np.float32)       # 2 s de fade
out[:len(f)] *= f; out[-len(f):] *= f[::-1]
sf.write(f"{SAIDA}/fundo_demonstracao.wav", out.astype(np.float32), FS, subtype="PCM_16")
rms_db = 20 * np.log10(np.sqrt(np.mean(out.astype(np.float64) ** 2)))
json.dump({"maquinas": MAQUINAS, "particao": PARTICAO_USADA, "duracao_min": DURACAO_MIN, "fs": FS,
           "nivel_rms_dbfs": round(float(rms_db), 1), "semente": SEMENTE, "clipes": usados},
          open(f"{SAIDA}/fundo_demonstracao.json", "w"), indent=1, ensure_ascii=False)
print(f"Gravado: {SAIDA}/fundo_demonstracao.wav | RMS {rms_db:.1f} dBFS | pico {20 * np.log10(np.abs(out).max()):.1f} dBFS")

Gravado: /content/drive/MyDrive/dissertacao_tinyml/demonstracao/fundo_demonstracao.wav | RMS -26.7 dBFS | pico -1.0 dBFS


In [5]:
# 5. Escuta rápida (30 s) no Colab
try:
    from IPython.display import Audio, display
    display(Audio(out[:30 * FS], rate=FS))
except Exception as e:
    print("Prévia indisponível:", e)

## Uso na demonstração

1. Baixe o `fundo_demonstracao.wav` e toque em loop no alto-falante (celular ou notebook), na **posição fixa** anotada para o guia de replicação.
2. **Antes da sessão**, com o ventilador desligado (para medir só o fundo), ajuste o volume até o decibelímetro marcar **70 dB(A) no ponto do sensor**. Não mexa mais no volume.
3. Faça o **CAL sem o fundo** (só o ventilador).
4. Ligue o fundo no início do Evento 3 e desligue ao fim do Evento 4.

No texto (Apêndice B, item e, ou Tabela C1): fundo composto por gravações normais dos ventiladores id_02, id_04 e id_06 do MIMII, partição de teste (notebook 13).